# 📐 Fundamentos: Aprendizaje No Supervisado

## La idea, en una frase
> **Encontrar estructura en datos que nadie etiquetó**: qué observaciones se parecen entre sí (agrupamiento) y qué pocas direcciones resumen la información (reducción de dimensionalidad).

Para lograrlo necesitamos tres herramientas que usaremos en **todo** el curso:
1. Una forma de medir qué tan **parecidas** son dos observaciones → **distancias**
2. Poner todas las variables en la **misma escala** para que ninguna domine → **escalado**
3. Una forma de saber si un agrupamiento es **bueno** sin tener la respuesta correcta → **métricas**

## 🧭 Lo que vas a aprender
1. Distinguir un problema **supervisado** de uno **no supervisado** y reconocer sus tareas
2. Calcular distancias **euclidiana, Manhattan, coseno y Haversine**, a mano y con scikit-learn
3. Explicar **por qué hay que escalar** y cómo cambia el vecino más cercano con `StandardScaler` y `MinMaxScaler`
4. Evaluar un agrupamiento **sin etiquetas** (silhouette) y, cuando sí conocemos la respuesta, compararlo con ella en una **tabla cruzada**

## ⏱️ Agenda (2 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 5 min |
| 1 | Supervisado vs no supervisado | 15 min |
| 2 | Distancias: ¿qué tan parecidos son dos puntos? | 30 min |
| 3 | Por qué escalar | 25 min |
| ☕ | Descanso | 10 min |
| 4 | Cómo se evalúa algo sin etiquetas | 30 min |
| 5 | Errores comunes y cierre | 5 min |

---
# 0️⃣ Preparación (5 min)

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_silueta_k(tabla)` | Silhouette promedio según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [1]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido y silhouette (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y silhouette en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Silhouette de un agrupamiento. El ruido (-1) se excluye del cálculo,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
    else:
        fila['silhouette'] = np.nan
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

---
# 1️⃣ Supervisado vs no supervisado (15 min)

En el curso de Modelos Predictivos **siempre** había una columna que queríamos predecir (`mpg`, `precio`, `churn`…). Esa columna es la **respuesta deseada**: el modelo aprende comparando su predicción contra ella.

Imagina las lecturas de sensores de una máquina en una planta. Si el área de mantenimiento anotó cuándo falló, tenemos un problema **supervisado**:

In [ ]:
import numpy as np
import pandas as pd

maquina = pd.DataFrame({
    'Turno': ['Mañana', 'Mañana', 'Tarde', 'Tarde', 'Noche', 'Noche'],
    'RPM': [1510, 1795, 1488, 1240, 1820, 1265],
    'Temp (°C)': [64, 79, 66, 93, 77, 90],
    'Corriente (A)': [12.1, 16.3, 11.8, 19.4, 15.9, 18.7],
    'Fallo': ['No', 'No', 'No', 'Sí', 'No', 'Sí'],
})
maquina

Ahora imagina que **nadie anotó los fallos**. Es lo más común en la vida real: etiquetar cuesta tiempo, dinero o simplemente no se hizo. Nos quedan sólo las lecturas:

In [ ]:
# Sin la columna 'Fallo': ya no hay respuesta deseada
maquina_sin_etiqueta = maquina.drop(columns=['Fallo'])
maquina_sin_etiqueta

| | Supervisado | No supervisado |
|---|---|---|
| ¿Existe la respuesta deseada? | **Sí** (`Fallo`, `precio`, `churn`…) | **No** |
| Pregunta típica | ¿Fallará esta máquina? ¿Cuánto costará? | ¿Qué **tipos** de operación hay? ¿Qué lectura es **rara**? |
| Tareas | Clasificación, regresión | **Agrupamiento** (clustering), **reducción de dimensionalidad** |
| ¿Cómo sé si acerté? | Comparo contra la etiqueta (R², accuracy…) | No hay contra qué comparar: **métricas internas** + sentido de negocio |

Veámoslo con 300 lecturas simuladas de la misma máquina. A la izquierda, lo que veríamos **si** tuviéramos la etiqueta; a la derecha, lo que **realmente** tenemos:

In [ ]:
# 300 lecturas simuladas: operación normal, carga alta y operación previa a un fallo
rng = np.random.default_rng(42)

def simular_regimen(n, rpm, temp, regimen):
    return pd.DataFrame({
        'RPM': rng.normal(rpm[0], rpm[1], n),
        'Temp (°C)': rng.normal(temp[0], temp[1], n),
        'regimen': regimen,
    })

lecturas = pd.concat([
    simular_regimen(180, (1500, 40), (65, 3), 'Normal'),
    simular_regimen(90, (1800, 40), (78, 3), 'Carga alta'),
    simular_regimen(30, (1250, 50), (92, 3), 'Previo a fallo'),
], ignore_index=True)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=['Con etiqueta (supervisado)', 'Sin etiqueta (no supervisado)']
)
for regimen in lecturas['regimen'].unique():
    sub = lecturas[lecturas['regimen'] == regimen]
    fig.add_trace(go.Scatter(x=sub['RPM'], y=sub['Temp (°C)'], mode='markers', name=regimen,
                             marker={'size': 6, 'opacity': 0.7}), row=1, col=1)
fig.add_trace(go.Scatter(x=lecturas['RPM'], y=lecturas['Temp (°C)'], mode='markers', name='Sin etiqueta',
                         marker={'size': 6, 'opacity': 0.7, 'color': 'gray'}), row=1, col=2)
fig.update_xaxes(title_text='RPM')
fig.update_yaxes(title_text='Temperatura (°C)')
fig.update_layout(width=1050, height=450, title='La misma máquina, con y sin la respuesta deseada')
fig.show()

✅ **Qué observar:**
- A la derecha **no sabemos** qué punto es qué, pero a simple vista se ven **3 nubes**: el ojo ya está haciendo *clustering*
- La nube pequeña de arriba a la izquierda (pocas RPM, mucha temperatura) es justo la que más le importaría a mantenimiento: un algoritmo podría **descubrirla sin que nadie la etiquete**
- Con 2 variables lo vemos a ojo. Con 20 sensores ya no: por eso necesitamos algoritmos (y por eso existe la **reducción de dimensionalidad**, para volver a verlo en 2D)

### Las dos tareas del curso

| Tarea | Qué hace | Ejemplo con la máquina | Algoritmos del curso |
|---|---|---|---|
| **Agrupamiento** (*clustering*) | Asigna una etiqueta a cada observación según su **similitud** con las demás | Descubrir los regímenes de operación | **K-Means** (sesión 1), **DBSCAN** (sesión 2) |
| **Reducción de dimensionalidad** | Expresa los mismos datos con **menos columnas**, conservando la tendencia de interés ($X_1, X_2 \rightarrow Z_1$) | Resumir 20 sensores en 2 o 3 "indicadores" para graficarlos | **PCA** (sesión 3) |

### ¿Para qué se usa en la industria?

| Caso de uso | Pregunta | Técnica típica |
|---|---|---|
| **Segmentación de clientes** | ¿Qué tipos de clientes tengo y qué campaña le toca a cada uno? | K-Means |
| **Detección de anomalías** | ¿Qué transacción, lectura o cliente no se parece a ningún otro? | DBSCAN (puntos de ruido) |
| **Geoanalítica** | ¿Dónde se concentran las entregas, los accidentes o los tramos más transitados de una mina? | DBSCAN con Haversine |
| **Compresión y visualización** | ¿Cuántas variables necesito realmente? ¿Cómo grafico 13 dimensiones? | PCA |

💡 Todo lo anterior depende de una misma pregunta: **¿qué tan parecidas son dos observaciones?** Eso es una **distancia**.

---
# 2️⃣ Distancias: ¿qué tan parecidos son dos puntos? (30 min)

Para un algoritmo, "parecido" significa "**cerca**". Pero hay muchas formas de medir "cerca". Usemos dos puntos sencillos, $A = (2, 1)$ y $B = (5, 5)$:

| Distancia | Fórmula | Intuición |
|---|---|---|
| **Euclidiana** | $d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$ | La línea recta, "a vuelo de pájaro" |
| **Manhattan** | $d(p, q) = \sum_{i=1}^{n} \lvert p_i - q_i \rvert$ | Caminar por cuadras: sólo te mueves en horizontal o vertical |
| **Coseno** | $d(p, q) = 1 - \dfrac{p \cdot q}{\lVert p \rVert \, \lVert q \rVert}$ | Compara la **dirección** de los vectores, no su tamaño |

In [ ]:
A = np.array([2, 1])
B = np.array([5, 5])

# A mano, con numpy
d_euclidiana = np.sqrt(np.sum((A - B) ** 2))
d_manhattan = np.sum(np.abs(A - B))
d_coseno = 1 - (A @ B) / (np.linalg.norm(A) * np.linalg.norm(B))

print(f"Euclidiana: {d_euclidiana:.3f}")
print(f"Manhattan:  {d_manhattan:.3f}")
print(f"Coseno:     {d_coseno:.3f}")

En la práctica no las programamos: `sklearn.metrics.pairwise` las calcula entre **todas** las parejas de filas de dos matrices. Ojo: esperan matrices 2D (una fila por observación), por eso usamos `[A]` y `[B]`.

In [ ]:
from sklearn.metrics.pairwise import (
    cosine_distances,
    euclidean_distances,
    haversine_distances,
    manhattan_distances,
)

print(f"Euclidiana: {euclidean_distances([A], [B])[0, 0]:.3f}")
print(f"Manhattan:  {manhattan_distances([A], [B])[0, 0]:.3f}")
print(f"Coseno:     {cosine_distances([A], [B])[0, 0]:.3f}")

In [ ]:
# Las dos rutas de A a B: línea recta (euclidiana) y por cuadras (Manhattan)
fig = go.Figure()
fig.add_trace(go.Scatter(x=[2, 5], y=[1, 5], mode='lines', name=f'Euclidiana = {d_euclidiana:.0f}',
                         line={'width': 3}))
fig.add_trace(go.Scatter(x=[2, 5, 5], y=[1, 1, 5], mode='lines', name=f'Manhattan = {d_manhattan:.0f}',
                         line={'width': 3, 'dash': 'dash'}))
fig.add_trace(go.Scatter(x=[2, 5], y=[1, 5], mode='markers+text', text=['A', 'B'],
                         textposition='top left', marker={'size': 14, 'color': 'black'}, showlegend=False))
fig.update_layout(width=600, height=500, title='Dos formas de medir la distancia de A a B',
                  xaxis={'range': [0, 6], 'dtick': 1}, yaxis={'range': [0, 6], 'dtick': 1, 'scaleanchor': 'x'})
fig.show()

✅ **Qué observar:**
- La euclidiana es **5** (el famoso triángulo 3-4-5) y la Manhattan es **3 + 4 = 7**: por cuadras siempre caminas igual o más
- La distancia coseno es casi **0 (0.051)**: los vectores $A$ y $B$ apuntan **casi en la misma dirección** desde el origen, aunque estén lejos
- `numpy` a mano y `sklearn` dan **exactamente** lo mismo

### ¿Cuándo usar coseno? Cuando importa la **proporción**, no el tamaño

Piensa en dos clientes de un súper que compran en 4 categorías (frutas, lácteos, vinos, limpieza). El cliente 2 compra **exactamente lo mismo** que el 1, pero **10 veces más**. ¿Son "parecidos"?

In [ ]:
# Unidades compradas por categoría: frutas, lácteos, vinos, limpieza
clientes = np.array([
    [2, 1, 0, 4],      # cliente 1
    [20, 10, 0, 40],   # cliente 2: mismas proporciones que el 1, pero 10 veces más
    [4, 2, 3, 0],      # cliente 3: volumen parecido al 1, pero otros gustos
])

pd.DataFrame({
    'euclidiana_vs_cliente_1': euclidean_distances(clientes, clientes[:1])[:, 0].round(2),
    'coseno_vs_cliente_1': cosine_distances(clientes, clientes[:1])[:, 0].round(2),
}, index=['cliente 1', 'cliente 2', 'cliente 3'])

✅ **Qué observar:**
- Con **euclidiana**, el cliente 3 (5.48) está mucho más cerca del 1 que el cliente 2 (41.24): sólo mira **cuánto** compran
- Con **coseno**, el cliente 2 es **idéntico** al 1 (distancia 0) y el 3 es el diferente (0.59): mira **qué** compran
- 💡 Ninguna es "la correcta": depende de la pregunta. Para *perfiles de gusto* (textos, canastas de compra) se usa coseno; para medidas físicas, euclidiana

### Haversine: distancias sobre la Tierra 🌎

Las coordenadas GPS **no** son un plano: son ángulos sobre una esfera. Un grado de longitud mide unos 111 km en el ecuador pero cada vez menos hacia los polos. La distancia **Haversine** mide el arco sobre la superficie:

$$a = \sin^2\!\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\!\left(\frac{\Delta \lambda}{2}\right)$$

$$d = 2R \cdot \arcsin\!\left(\sqrt{a}\right)$$

donde $\phi$ es la **latitud**, $\lambda$ la **longitud** (las dos **en radianes**) y $R \approx 6371$ km es el radio de la Tierra.

Calculemos la distancia entre **UDEM** (San Pedro Garza García) y la **Macroplaza** (centro de Monterrey):

In [ ]:
R_TIERRA_KM = 6371

udem = (25.6597, -100.4194)        # (latitud, longitud) aprox.
macroplaza = (25.6694, -100.3097)

def haversine_km(p1, p2):
    """Distancia Haversine en km entre dos puntos (lat, lon) en grados."""
    lat1, lon1, lat2, lon2 = np.radians([p1[0], p1[1], p2[0], p2[1]])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R_TIERRA_KM * np.arcsin(np.sqrt(a))

print(f"A mano (numpy): {haversine_km(udem, macroplaza):.2f} km")

# Con sklearn: espera [latitud, longitud] EN RADIANES y regresa la distancia en radianes
d_rad = haversine_distances(np.radians([udem]), np.radians([macroplaza]))[0, 0]
print(f"Con sklearn:    {d_rad * R_TIERRA_KM:.2f} km  (= {d_rad:.5f} rad × 6371)")

In [ ]:
# ¿Y si usamos euclidiana sobre los grados, como si fuera un plano?
d_grados = euclidean_distances([udem], [macroplaza])[0, 0]
print(f"Euclidiana en grados: {d_grados:.4f} grados  → × 111 km = {d_grados * 111:.2f} km")

In [ ]:
# Los dos puntos en el mapa
fig = go.Figure(go.Scattermap(
    lat=[udem[0], macroplaza[0]],
    lon=[udem[1], macroplaza[1]],
    mode='markers+lines+text',
    text=['UDEM', 'Macroplaza'],
    textposition='top right',
    marker={'size': 14},
))
fig.update_layout(
    map={'style': 'open-street-map', 'center': {'lat': 25.665, 'lon': -100.365}, 'zoom': 11},
    width=800,
    height=450,
    margin={'l': 0, 'r': 0, 't': 40, 'b': 0},
    title=f'UDEM → Macroplaza: {haversine_km(udem, macroplaza):.1f} km en línea recta sobre la Tierra'
)
fig.show()

✅ **Qué observar:**
- De UDEM a la Macroplaza hay **11.05 km** en línea recta sobre la Tierra; numpy y sklearn coinciden
- La euclidiana "en grados" da **0.1101 grados**, un número sin unidades útiles. Si lo multiplicas por 111 km sale **12.22 km**: un error de más de 1 km en una distancia de 11, porque en Monterrey un grado de longitud mide unos 100 km, no 111
- ⚠️ `haversine_distances` usa el orden **[latitud, longitud]** y **radianes**. Si le pasas grados no da error… da un número equivocado. En la sesión de DBSCAN usaremos esto mismo con `metric='haversine'` y `eps = km / 6371`

### ✍️ Mini-ejercicio 1: ¿qué tan lejos queda el aeropuerto?

El Aeropuerto Internacional de Monterrey está aproximadamente en **(25.7785, -100.1069)**. Calcula la distancia en km desde UDEM **con sklearn**.

In [ ]:
aeropuerto = (25.7785, -100.1069)

# 1. Convierte udem y aeropuerto a radianes   → np.radians([udem]), np.radians([aeropuerto])
# 2. Calcula la distancia con haversine_distances(...)[0, 0]  (sale en radianes)
# 3. Multiplícala por R_TIERRA_KM y guárdala en d_aeropuerto_km
# 4. Comprueba tu resultado con haversine_km(udem, aeropuerto)
# Pista: es exactamente lo que hicimos con la Macroplaza
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántos km hay de UDEM al aeropuerto? ¿Cuántas veces la distancia a la Macroplaza? ¿Será igual la distancia que marca Google Maps en auto? ¿Por qué?

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ Por qué escalar (25 min)

Las distancias suman diferencias de **todas** las variables. ¿Qué pasa si una variable se mide en años (decenas) y otra en pesos (decenas de miles)?

Dos personas: la **persona 1** tiene **45 años** y gana **\$50,000**; la **persona 2** tiene **30 años** y gana **\$75,000**.

In [ ]:
# Dos personas descritas por (edad, salario)
persona_1 = np.array([45, 50000])   # edad 45 años, salario $50,000
persona_2 = np.array([30, 75000])   # edad 30 años, salario $75,000

diferencias = np.abs(persona_1 - persona_2)
distancia_original = np.sqrt(np.sum((persona_1 - persona_2) ** 2))

print(f"Diferencia en edad:    {diferencias[0]} años")
print(f"Diferencia en salario: {diferencias[1]:,} pesos")
print(f"Distancia euclidiana:  {distancia_original:,.4f}")

🤔 La distancia es **25,000.0045**: prácticamente la diferencia de salario. Los **15 años** de diferencia aportan sólo **0.0045**. Para el algoritmo, la edad **no existe**.

No es que el salario sea más importante: es que sus **unidades** son más grandes. Si midiéramos el salario en miles de pesos, de pronto la edad dominaría. Un buen análisis no puede depender de la unidad que eligió quien capturó los datos.

### La solución: poner todas las variables en la misma escala

| Escalador | Fórmula | Resultado | Cuidado con |
|---|---|---|---|
| `StandardScaler` | $z = \dfrac{x - \mu}{\sigma}$ | Media 0 y desviación estándar 1 | Nada especial: es el **default** del curso |
| `MinMaxScaler` | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ | Todo entre 0 y 1 | Un solo **valor atípico** aplasta a todos los demás |

Apliquemos Min-Max a mano a las dos personas. Con sólo dos puntos, el mínimo y el máximo de cada variable son esos mismos valores (en un caso real se calculan sobre **todo** el conjunto de datos):

In [ ]:
# Mínimo y máximo de cada columna (aquí, sólo de estas dos personas)
dos_personas = np.vstack([persona_1, persona_2])
minimo = dos_personas.min(axis=0)   # [30, 50000]
maximo = dos_personas.max(axis=0)   # [45, 75000]

dos_personas_escaladas = (dos_personas - minimo) / (maximo - minimo)
print("Escaladas (edad, salario):")
print(dos_personas_escaladas)

distancia_escalada = np.sqrt(np.sum((dos_personas_escaladas[0] - dos_personas_escaladas[1]) ** 2))
print(f"Distancia euclidiana escalada: {distancia_escalada:.4f}")

✅ **Qué observar:**
- La persona 1 queda en **(1, 0)**: es la de **mayor** edad y **menor** salario. La persona 2 queda en **(0, 1)**
- La distancia es **1.4142** ($\sqrt{1^2 + 1^2}$): ahora edad y salario **pesan lo mismo**

### ¿Cambia el vecino más cercano? Un ejemplo con 8 personas

Esto no es un detalle estético. Los algoritmos de clustering deciden **quién va con quién** según las distancias. Busquemos a la persona más parecida a **Ana** (25 años, \$30,000):

In [ ]:
personas = pd.DataFrame({
    'edad': [25, 27, 58, 40, 33, 62, 22, 47],
    'salario': [30000, 42000, 31000, 60000, 25000, 80000, 18000, 50000],
}, index=['Ana', 'Beto', 'Carla', 'Diego', 'Elena', 'Fer', 'Gaby', 'Hugo'])
personas

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Distancia de cada persona a Ana, sin escalar
dist_sin_escalar = pd.Series(
    euclidean_distances(personas, personas.loc[['Ana']])[:, 0],
    index=personas.index
)

# Distancia de cada persona a Ana, con StandardScaler
scaler_std = StandardScaler()
personas_std = scaler_std.fit_transform(personas)
dist_std = pd.Series(
    euclidean_distances(personas_std, personas_std[:1])[:, 0],
    index=personas.index
)

print(f"Media aprendida:      edad = {scaler_std.mean_[0]:.2f} años | salario = ${scaler_std.mean_[1]:,.2f}")
print(f"Desviación aprendida: edad = {scaler_std.scale_[0]:.2f} años | salario = ${scaler_std.scale_[1]:,.2f}")
pd.DataFrame({
    'sin_escalar': dist_sin_escalar.round(1),
    'standard': dist_std.round(3),
}).drop(index='Ana').sort_values('sin_escalar')

✅ **Qué observar:**
- **Sin escalar**, la persona más parecida a Ana es **Carla** (distancia 1,000.5): ganan casi lo mismo, pero Carla tiene **58 años** y Ana 25. ¡33 años de diferencia "no cuentan"!
- **Con `StandardScaler`**, Carla se va al **penúltimo lugar** (2.32) y la más parecida pasa a ser **Elena** (0.619): 33 años y \$25,000
- `StandardScaler` aprendió una media de **39.25 años** y **\$42,000**, con desviaciones de **14.23 años** y **\$19,241.88**: cada variable se mide ahora "en desviaciones estándar", sin unidades
- 💡 El mismo algoritmo con los mismos datos da **otro resultado** sólo por escalar. Por eso en este curso **siempre** escalamos antes de K-Means, DBSCAN o PCA

### ✍️ Mini-ejercicio 2: ¿y con `MinMaxScaler`?

Repite el cálculo de distancias a Ana, pero escalando con `MinMaxScaler`. ¿Quién es ahora su vecino más cercano?

In [ ]:
# 1. Crea scaler_mm = MinMaxScaler() y aplica fit_transform a personas   → personas_mm
# 2. Calcula la distancia de cada persona a Ana (la primera fila):
#    euclidean_distances(personas_mm, personas_mm[:1])[:, 0]   → guárdala en una pd.Series dist_mm con index=personas.index
# 3. Arma una tabla con las columnas 'sin_escalar', 'standard' y 'minmax', quita a Ana y ordénala por 'minmax'
# Pista: copia la celda de StandardScaler y cambia el escalador
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Quién es el vecino más cercano de Ana con `MinMaxScaler`? ¿Coincide con `StandardScaler`? ¿En qué coinciden los dos escaladores y en qué no?

RELLENAR CON TUS COMENTARIOS

### ⚠️ El talón de Aquiles de Min-Max: los valores atípicos

Si a la tabla llega una persona que gana \$400,000, Min-Max usa ese valor como máximo y **aplasta** a todos los demás cerca de 0:

In [ ]:
con_atipico = pd.concat([personas, pd.DataFrame({'edad': [50], 'salario': [400000]}, index=['Iván'])])

salario_escalado = pd.DataFrame({
    'salario': con_atipico['salario'],
    'minmax': MinMaxScaler().fit_transform(con_atipico[['salario']])[:, 0].round(3),
    'standard': StandardScaler().fit_transform(con_atipico[['salario']])[:, 0].round(3),
})
salario_escalado

✅ **Qué observar:**
- Con Iván en la tabla, los 8 salarios originales quedan entre **0** y **0.162** con Min-Max: las diferencias entre ellos casi desaparecen
- `StandardScaler` también se ve afectado (la media y la desviación suben), pero no confina a nadie a un rango fijo
- 💡 Regla práctica del curso: **`StandardScaler` por defecto**. Si hay variables muy sesgadas o con atípicos (montos de dinero, conteos), considera antes una transformación logarítmica: lo haremos en el caso integrador

---
# ☕ Descanso (10 min)

---
# 4️⃣ Cómo se evalúa algo sin etiquetas (30 min)

En Modelos Predictivos comparábamos la predicción contra la respuesta real. Aquí **no hay respuesta**. ¿Cómo sabemos si un agrupamiento es bueno?

La idea: un buen agrupamiento tiene grupos **compactos** (los puntos de un grupo están cerca entre sí) y **separados** (lejos de los otros grupos). Las **métricas internas** miden eso usando sólo $X$ y las etiquetas que produjo el algoritmo. Hay varias (Davies-Bouldin y Calinski-Harabasz son otras dos conocidas), pero en este curso usaremos **una sola**: la **silhouette**, porque es la más fácil de interpretar y se puede ver punto por punto.

Para practicar usamos datos sintéticos de `make_blobs` donde **sí** sabemos la verdad: 500 puntos en **4 grupos**. Agrupamos con K-Means (lo estudiaremos a fondo en la sesión 1; por ahora es una caja negra que recibe el número de grupos `k`).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

X_blobs, y_blobs = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=1.0,
    random_state=42
)
blobs = pd.DataFrame(X_blobs, columns=['x1', 'x2'])

# Un modelo con el k correcto y otro con un k incorrecto
kmeans_4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_blobs)
kmeans_2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_blobs)

plot_clusters(blobs, 'x1', 'x2', kmeans_4.labels_, centroides=kmeans_4.cluster_centers_, title='K-Means con k = 4')
plot_clusters(blobs, 'x1', 'x2', kmeans_2.labels_, centroides=kmeans_2.cluster_centers_, title='K-Means con k = 2')

### Silhouette: ¿cada punto está en el grupo correcto?

Para **cada punto** $i$:
- $a(i)$ = distancia promedio a los puntos de **su propio** cluster (¿qué tan bien encaja en su casa?)
- $b(i)$ = distancia promedio a los puntos del cluster **vecino más cercano** (¿qué tan lejos está la casa de al lado?)

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}} \qquad -1 \le s(i) \le 1$$

| $s(i)$ | Significado |
|---|---|
| Cerca de **1** | $a \ll b$: el punto está bien metido en su grupo y lejos de los demás |
| Cerca de **0** | $a \approx b$: está en la frontera entre dos grupos |
| **Negativo** | $a > b$: estaría mejor en el otro grupo 🚩 |

El **silhouette** de un modelo es el promedio de $s(i)$ sobre todos los puntos. Calculémoslo a mano para el primer punto:

In [ ]:
from sklearn.metrics import silhouette_samples

i = 0
labels_4 = kmeans_4.labels_
distancias_i = euclidean_distances(X_blobs[[i]], X_blobs)[0]

# a(i): promedio a los de su mismo cluster (sin contarse a sí mismo)
mismo = (labels_4 == labels_4[i])
mismo[i] = False
a_i = distancias_i[mismo].mean()

# b(i): el menor promedio hacia cada uno de los otros clusters
b_i = min(distancias_i[labels_4 == c].mean() for c in np.unique(labels_4) if c != labels_4[i])

s_i = (b_i - a_i) / max(a_i, b_i)
print(f"a(i) = {a_i:.3f} | b(i) = {b_i:.3f} | s(i) = {s_i:.3f}")
print(f"sklearn (silhouette_samples): {silhouette_samples(X_blobs, labels_4)[i]:.3f}")

✅ **Qué observar:**
- El punto 0 está, en promedio, a **1.699** de los de su grupo y a **5.738** del grupo vecino: $s = 0.704$, bien asignado
- Nuestro cálculo a mano coincide con `silhouette_samples` de sklearn

Ahora veamos la silueta de **todos** los puntos con el k correcto y con el incorrecto:

In [ ]:
plot_silueta(X_blobs, kmeans_4.labels_)
plot_silueta(X_blobs, kmeans_2.labels_)

✅ **Qué observar:**
- Con **k = 4**, las 4 "cuchillas" tienen el mismo tamaño (125 puntos cada una), el **97%** de los puntos está por arriba de 0.6 y el promedio es **0.791**
- Con **k = 2**, el promedio baja a **0.596**: un cluster tiene **375 puntos** (metimos **tres** grupos reales en uno solo) y su silueta promedio cae a 0.500, mientras el otro (125 puntos) queda en 0.884

### Una tabla para comparar modelos

La función `evaluar_clustering` calcula la silhouette de un modelo en una línea (y el % de ruido, que usaremos con DBSCAN):

In [ ]:
comparacion = []
comparacion.append(evaluar_clustering('K-Means k=4 (correcto)', X_blobs, kmeans_4.labels_))
comparacion.append(evaluar_clustering('K-Means k=2 (incorrecto)', X_blobs, kmeans_2.labels_))
pd.DataFrame(comparacion)

✅ **Qué observar:**
- k = 4 tiene mejor silhouette (**0.791** vs 0.596): la tabla dice lo mismo que vimos en las gráficas de silueta
- ⚠️ Aquí todo es fácil porque los grupos son redondos y bien separados. Con datos reales la silhouette a veces no marca un ganador claro: ahí entran el codo y el sentido de negocio (lo veremos con los clientes del centro comercial)

### Cuando **sí** tenemos la etiqueta: la tabla cruzada

En datos de práctica (como `make_blobs`, Iris o Wine) conocemos la etiqueta real. Para ver si el agrupamiento la recupera, basta una **tabla cruzada**: cada renglón es un grupo real, cada columna un cluster, y cada celda cuenta cuántos puntos caen ahí.
- Lo que importa es **quién va con quién**, no el nombre: el "cluster 0" de K-Means no tiene por qué ser el "grupo 0" real.

In [ ]:
print("k = 4 contra la etiqueta real:")
display(pd.crosstab(y_blobs, kmeans_4.labels_, rownames=['grupo_real'], colnames=['cluster']))

print("k = 2 contra la etiqueta real:")
display(pd.crosstab(y_blobs, kmeans_2.labels_, rownames=['grupo_real'], colnames=['cluster']))

✅ **Qué observar:**
- Con k = 4, cada grupo real cae **completo** en un solo cluster (125 de 125) y cada cluster tiene un solo grupo real: K-Means recupera **exactamente** la verdad, aunque con otros nombres (el grupo real 0 es el cluster 3)
- Con k = 2, el cluster 0 **mezcla tres grupos reales** (125 + 125 + 125 = 375 puntos); sólo el grupo real 2 queda solo en el cluster 1
- ⚠️ En un proyecto real **no hay** `y_blobs`: si tuvieras la etiqueta, probablemente usarías un modelo supervisado. La tabla cruzada contra la verdad sirve para **aprender y validar métodos**, no para elegir k en producción

### ✍️ Mini-ejercicio 3: ¿y si nos pasamos de grupos?

Ajusta K-Means con **k = 6** (un k incorrecto "por arriba"), agrégalo a la tabla `comparacion` y haz su tabla cruzada contra `y_blobs`.

In [ ]:
# 1. kmeans_6 = KMeans(n_clusters=6, n_init=10, random_state=42).fit(X_blobs)
# 2. Grafícalo con plot_clusters(blobs, 'x1', 'x2', kmeans_6.labels_, centroides=kmeans_6.cluster_centers_)
# 3. Agrega su fila: comparacion.append(evaluar_clustering('K-Means k=6 (incorrecto)', X_blobs, ...))
# 4. Muestra la tabla de comparación: display(pd.DataFrame(comparacion))
# 5. Haz la tabla cruzada: pd.crosstab(y_blobs, kmeans_6.labels_, rownames=['grupo_real'], colnames=['cluster'])
# ✍️ Tu código aquí


✍️ **¿Qué observas?** Compara k = 6 contra k = 4 y contra k = 2 en la silhouette. ¿Qué hace k = 6 con los grupos reales según la tabla cruzada: los parte o los mezcla? ¿Y k = 2? ¿Qué te parece "menos malo": pasarse de grupos o quedarse corto? ¿La silhouette opina lo mismo?

RELLENAR CON TUS COMENTARIOS

---
# 5️⃣ ⚠️ Errores comunes (5 min)

| Error | Por qué es un problema | Qué hacer |
|---|---|---|
| **No escalar** antes de K-Means, DBSCAN o PCA | La variable con unidades más grandes decide todo (el salario se "come" a la edad) | `StandardScaler` por defecto |
| Usar `MinMaxScaler` con **valores atípicos** | Un solo extremo aplasta al resto cerca de 0 | `StandardScaler` o transformación log antes |
| Haversine con **grados** o en orden **(lon, lat)** | No marca error: da un número equivocado | `np.radians([[lat, lon]])` y multiplicar por 6371 para km |
| Euclidiana directa sobre lat/lon | Un grado de longitud no mide lo mismo en todas partes | Haversine |
| Creer que el **"cluster 0" es la "clase 0"** | Los números de cluster son arbitrarios | Con datos de práctica, comparar con una **tabla cruzada**: importa quién va con quién, no el número |
| Elegir k **sólo** con la silhouette | Un promedio puede esconder clusters malos o empates | Silhouette + codo + gráfica de silueta **y** sentido de negocio |
| Esperar una "respuesta correcta" en un problema real | En la práctica no existe la etiqueta | La tabla cruzada contra la verdad es sólo para validar métodos con datos de práctica |

---
# 📝 Lo que aprendimos

1. **No supervisado** = la respuesta deseada **no existe**. Las dos tareas del curso son **agrupamiento** (K-Means, DBSCAN) y **reducción de dimensionalidad** (PCA)
2. "Parecido" = "cerca", y hay varias formas de medirlo: **euclidiana** (línea recta), **Manhattan** (por cuadras), **coseno** (dirección/proporciones) y **Haversine** (sobre la Tierra: de UDEM a la Macroplaza hay 11.05 km)
3. **Escalar no es opcional**: sin escalar, el vecino más cercano de Ana era alguien 33 años mayor sólo porque gana parecido
4. Sin etiquetas evaluamos **compacidad y separación** con la **silhouette** (de -1 a 1, más alto es mejor). Con el k correcto sube a 0.791
5. Cuando tenemos la respuesta real (datos de práctica), la comparamos con una **tabla cruzada**: k = 6 sólo partía grupos, k = 2 mezclaba tres. En la vida real no la tenemos

### 🗺️ Lo que sigue
| Sesión | Tema | Pregunta |
|---|---|---|
| 1 | **K-Means** | ¿Qué segmentos de clientes tiene el centro comercial? |
| 2 | **DBSCAN** | ¿Qué clientes no encajan en ningún segmento? ¿Dónde poner centros de distribución? |
| 3 | **PCA** | ¿Cuántas medidas químicas necesitamos para distinguir vinos? |
| 4 | **Caso integrador** | ¿Cómo debe organizar el distribuidor a sus clientes mayoristas? |